# 📈 LONQ — Sistema de detección de regímenes de mercado
> **Detecta automáticamente si el mercado está en régimen Bull, Bear o Sideways**  
> usando un ensemble de K-Means + GMM + HMM, y genera señales de compra/venta.

---
### ¿Qué hace este notebook?
1. Descarga datos históricos de cualquier ticker (vía yfinance)
2. Calcula 13 features técnicas (volatilidad, momentum, RSI, etc.)
3. Entrena 3 modelos y combina su consenso
4. Genera señales **BUY / SELL / HOLD**
5. Hace backtest vs Buy & Hold
6. Muestra gráficos interactivos

⏱️ Tiempo estimado: ~2-3 minutos para correr todo


## ⚙️ Paso 1 — Instalar dependencias y clonar el repo

In [ ]:
# Instalar librerías necesarias
!pip install hmmlearn scikit-learn matplotlib pandas numpy scipy yfinance -q
print("✓ Dependencias instaladas")

In [ ]:
# Clonar el repositorio LONQ (rama con todas las funcionalidades)
import os

BRANCH = "claude/continue-lonq-7VADd"

if not os.path.exists("A_tecn"):
    !git clone --branch {BRANCH} https://github.com/pfreezv/A_tecn.git
    print("✓ Repositorio clonado")
else:
    !cd A_tecn && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
    print("✓ Repositorio actualizado")

os.chdir("A_tecn")
import sys
sys.path.insert(0, ".")

# Verificar que src/signals.py existe
import os.path
assert os.path.exists("src/signals.py"), "ERROR: src/signals.py no encontrado — verifica la rama"
print("✓ Listo — todos los módulos disponibles")

## 🎯 Paso 2 — Configuración
> **Cambia el ticker aquí para analizar cualquier acción**

In [ ]:
# ════════════════════════════════════════════════
#   CONFIGURACIÓN — edita estos valores
# ════════════════════════════════════════════════

TICKER     = "KO"          # Ticker a analizar (KO=Coca-Cola, AAPL, TSLA, SPY, BTC-USD...)
START_DATE = "2020-01-01"  # Fecha de inicio
K_MIN      = 2             # Número mínimo de clusters a probar
K_MAX      = 5             # Número máximo de clusters a probar
SHOW_PLOTS = True          # True = mostrar gráficos

print(f"Configurado: {TICKER}  |  desde {START_DATE}  |  K={K_MIN}–{K_MAX}")

## 📥 Paso 3 — Descargar datos

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

from src.data import fetch_ohlcv

raw = fetch_ohlcv(TICKER, start_date=START_DATE)
print(f"✓ {TICKER}: {len(raw)} días cargados")
print(f"  Rango: {raw.index[0].date()} → {raw.index[-1].date()}")
print(f"  Precio último cierre: ${raw['Close'].iloc[-1]:.2f}")
raw.tail(3)

## 🤖 Paso 4 — Ensemble de regímenes (K-Means + GMM + HMM)

In [ ]:
from src.ensemble import fit_ensemble

result = fit_ensemble(TICKER, raw, k_min=K_MIN, k_max=K_MAX, show_progress=True)

# Resumen
print("\n--- Distribución de regímenes ---")
dist = result.consensus.value_counts(normalize=True).mul(100).round(1)
for reg, pct in dist.items():
    bar = "█" * int(pct / 2)
    print(f"  {reg:10s} {pct:5.1f}%  {bar}")

print("\n--- Métricas por modelo ---")
print(result.metrics.round(3).to_string())

## 🔔 Paso 5 — Señales de compra y venta

In [ ]:
from src.signals import generate_signals, print_signals

sig_result = generate_signals(
    consensus  = result.consensus,
    confidence = result.confidence,
    prices     = result.df["Close"],
    buy_threshold  = 0.67,   # 2 de 3 modelos deben coincidir para BUY
    sell_threshold = 0.33,   # cualquier modelo puede disparar SELL
)

print_signals(sig_result)

In [ ]:
# Ver log completo de operaciones
if not sig_result.trades.empty:
    print("\nHistorial de operaciones:")
    display(sig_result.trades)
else:
    print("No se generaron operaciones en este período")

## 📊 Paso 6 — Backtest régimen vs Buy & Hold

In [ ]:
from src.backtest import run_backtest, run_buyhold, compare_strategies

strat = run_backtest(raw["Close"], result.consensus, strategy_name=f"Regime {TICKER}")
bh    = run_buyhold(raw["Close"])

print("\n=== Comparación de estrategias ===")
display(compare_strategies([strat, bh]))

## 📈 Paso 7 — Visualizaciones

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

df   = result.df
sigs = sig_result.signals
COLORS = {"bull": "#2ecc71", "sideways": "#f39c12", "bear": "#e74c3c"}

fig, axes = plt.subplots(4, 1, figsize=(15, 18))
fig.suptitle(f"LONQ — {TICKER}", fontsize=16, fontweight="bold", y=0.98)

# ── 1. Precio + regímenes + señales ──
ax = axes[0]
ax.plot(df.index, df["Close"], color="#2c3e50", lw=1, zorder=3, label="Precio")
prev_d, prev_r = df.index[0], result.consensus.iloc[0]
for d, r in zip(df.index[1:], result.consensus.iloc[1:]):
    if r != prev_r or d == df.index[-1]:
        ax.axvspan(prev_d, d, alpha=0.18, color=COLORS.get(prev_r, "gray"))
        prev_d, prev_r = d, r

buys  = sigs[sigs["Signal"] == "BUY"]
sells = sigs[sigs["Signal"] == "SELL"]
ax.scatter(buys.index,  buys["Close"],  marker="^", color="#27ae60", s=100, zorder=5, label="BUY")
ax.scatter(sells.index, sells["Close"], marker="v", color="#c0392b", s=100, zorder=5, label="SELL")
patches = [mpatches.Patch(facecolor=c, alpha=0.4, label=r) for r, c in COLORS.items()]
ax.legend(handles=patches + [
    plt.Line2D([0],[0], marker="^", color="w", markerfacecolor="#27ae60", markersize=10, label="BUY"),
    plt.Line2D([0],[0], marker="v", color="w", markerfacecolor="#c0392b", markersize=10, label="SELL"),
], fontsize=9, ncol=3, loc="upper left")
ax.set_title("Precio con regímenes detectados y señales")
ax.set_ylabel("Precio ($)")

# ── 2. Posición ──
ax = axes[1]
ax.fill_between(sigs.index, sigs["Position"], alpha=0.55, color="#2980b9", step="post")
ax.set_title("Posición del portfolio (1 = en mercado, 0 = en cash)")
ax.set_ylabel("Posición")
ax.set_ylim(-0.05, 1.3)

# ── 3. Equity curves ──
ax = axes[2]
ax.plot(strat.equity_curve.index, strat.equity_curve, label="Regime Strategy", color="#2980b9", lw=2)
ax.plot(bh.equity_curve.index,    bh.equity_curve,    label="Buy & Hold",      color="#e74c3c", lw=2, ls="--")
ax.axhline(1.0, color="gray", lw=0.5, ls=":")
ax.fill_between(strat.equity_curve.index,
                (strat.equity_curve / strat.equity_curve.cummax() - 1),
                alpha=0.15, color="#2980b9")
ax.set_title("Curva de capital")
ax.set_ylabel("Valor (base 1.0)")
ax.legend()

# ── 4. Confianza del ensemble ──
ax = axes[3]
conf = result.confidence
colors_conf = conf.map(lambda c: "#27ae60" if c == 1.0 else ("#f39c12" if c >= 0.67 else "#e74c3c"))
ax.bar(conf.index, conf, color=colors_conf, width=1, alpha=0.8)
ax.axhline(0.67, color="orange", lw=1, ls="--", label="Moderado")
ax.axhline(1.0,  color="green",  lw=1, ls="--", label="Fuerte")
ax.set_title("Confianza del ensemble (verde=fuerte, naranja=moderado, rojo=débil)")
ax.set_ylabel("Confianza")
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"lonq_{TICKER}.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"\n✓ Gráfico guardado como lonq_{TICKER}.png")

## 🔄 Paso 8 (opcional) — Walk-Forward
> Re-entrena el modelo cada 20 días simulando despliegue real. **Tarda ~2 minutos.**

In [ ]:
# Descomentar para ejecutar walk-forward
# from src.features import build_features, get_feature_matrix, FEATURE_COLS
# from src.walkforward import walk_forward
#
# df_feat  = build_features(raw)
# df_model = get_feature_matrix(df_feat)
# wf = walk_forward(df_model, FEATURE_COLS, k_kmeans=3, k_gmm=3, n_hmm=3)
#
# print("Distribución walk-forward:")
# print(wf["Consensus"].value_counts(normalize=True).mul(100).round(1).to_string())
# print(wf.tail(10).to_string())

## ✅ Resumen
| Componente | Descripción |
|---|---|
| **K-Means** | Agrupa días por similitud de features técnicas |
| **GMM** | Modelo probabilístico, captura regímenes con forma elíptica |
| **HMM** | Modelo de Markov oculto, captura transiciones entre estados |
| **Consenso** | Voto de mayoría de los 3 modelos |
| **BUY** | El consenso pasa a *bull* con ≥2 modelos de acuerdo |
| **SELL** | El consenso pasa a *bear* o sale de *bull* |

---
*LONQ — github.com/pfreezv/A_tecn*
